# Retail Data Preprocessing & Visualization

## Business Scenario
A retail chain operating across multiple cities in India wants to improve decision-making by analyzing customer purchase behavior. The raw transactional data is messy, incomplete, and inconsistent. The purpose of this project is to preprocess the data and create visualizations that can help understand customer preferences, seasonal demand, and city-level sales performance.

## Objective
Prepare a clean retail dataset and generate beginner-friendly visual insights on customer demographics, sales trends, city-wise revenue, category performance, and payment behavior.

## 1) Import Libraries

In [ ]:
import os
from pathlib import Path
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2) Load Dataset
This section identifies the project root safely, creates required output folders, validates required columns, and loads the dataset.

In [ ]:
current_dir = Path.cwd()
if (current_dir / "data").exists():
    project_root = current_dir
elif (current_dir.parent / "data").exists():
    project_root = current_dir.parent
else:
    project_root = current_dir

data_path = project_root / "data" / "retail_transactions_2000.csv"
cleaned_output_path = project_root / "data" / "Retail_Cleaned.csv"
viz_dir = project_root / "visualizations"
os.makedirs(viz_dir, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Raw dataset path: {data_path}")
print(f"Visualization output path: {viz_dir}")

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {data_path}. Please place retail_transactions_2000.csv in the data folder."
    )

df = pd.read_csv(data_path)

required_columns = [
    "TransactionID", "CustomerID", "Gender", "Age", "City", "ProductCategory",
    "Quantity", "Price", "TotalAmount", "PurchaseDate", "PaymentMode"
]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(
        f"Dataset is missing required columns: {missing_columns}. Please verify the CSV schema."
    )

print("Dataset loaded successfully.")

## 3) Data Inspection
**What:** Inspect shape, schema, summary, missing values, and duplicates.

**Why:** Helps understand quality issues before cleaning.

**What changed:** No data changed in this step; this is baseline inspection.

In [ ]:
print("\n========== FIRST 5 ROWS ==========")
display(df.head())

print("\n========== LAST 5 ROWS ==========")
display(df.tail())

print("\n========== SHAPE ==========")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n========== COLUMN NAMES ==========")
print(df.columns.tolist())

print("\n========== DATA TYPES ==========")
print(df.dtypes)

print("\n========== DATASET INFO ==========")
buffer = io.StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

print("\n========== DESCRIPTIVE STATISTICS ==========")
display(df.describe(include="all").transpose())

important_cats = ["Gender", "City", "ProductCategory", "PaymentMode"]
print("\n========== UNIQUE VALUE COUNTS ==========")
for col in important_cats:
    print(f"{col}: {df[col].nunique(dropna=True)} unique values")

print("\n========== MISSING VALUES ==========")
print(df.isna().sum())

print("\n========== DUPLICATE ROWS ==========")
print(df.duplicated().sum())

## 4) Missing Value Analysis
**What:** Fill non-critical missing values and remove rows with missing critical identifiers.

**Why:** Missing key IDs/categories can break reliability; `Age` and `City` can be safely imputed.

**What changed:** Missing values reduced using documented rules.

In [ ]:
print("Missing values before cleaning:")
print(df.isna().sum())

# Drop rows with critical missing identifiers
critical_fields = ["TransactionID", "ProductCategory"]
df = df.dropna(subset=critical_fields).copy()

# Handle Age
age_series = pd.to_numeric(df["Age"], errors="coerce")
age_fill_value = age_series.median()
df["Age"] = age_series.fillna(age_fill_value)

# Handle City
city_mode = df["City"].mode(dropna=True)
if not city_mode.empty:
    df["City"] = df["City"].fillna(city_mode.iloc[0])

print("\nMissing values after missing-value handling:")
print(df.isna().sum())

## 5) Duplicate Detection
**What:** Count and remove duplicated rows.

**Why:** Duplicate transactions can distort KPIs and trend analysis.

**What changed:** Duplicate records were removed.

In [ ]:
duplicates_before = df.duplicated().sum()
print(f"Duplicate rows before removal: {duplicates_before}")

df = df.drop_duplicates().copy()

duplicates_after = df.duplicated().sum()
print(f"Duplicate rows after removal: {duplicates_after}")

## 6) Data Cleaning (Categorical Standardization)
**What:** Clean spacing/casing and standardize key categorical labels.

**Why:** Inconsistent categories create fragmented analysis results.

**What changed:** Unified labels for `Gender` and cleaned categorical formatting.

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return np.nan
    return " ".join(str(value).strip().split())

for col in ["Gender", "City", "ProductCategory", "PaymentMode"]:
    df[col] = df[col].apply(clean_text)

# Standardize gender labels while preserving 'Other'
def normalize_gender(value):
    if pd.isna(value):
        return np.nan
    lower_val = value.lower()
    if lower_val in {"m", "male"}:
        return "Male"
    if lower_val in {"f", "female"}:
        return "Female"
    if lower_val in {"other", "o"}:
        return "Other"
    return value

df["Gender"] = df["Gender"].apply(normalize_gender)

# Standardize payment mode naming for common values
def normalize_payment_mode(value):
    if pd.isna(value):
        return np.nan
    lower_val = value.lower()
    mapping = {
        "cash": "Cash",
        "card": "Card",
        "upi": "UPI",
        "wallet": "Wallet"
    }
    return mapping.get(lower_val, value)

df["PaymentMode"] = df["PaymentMode"].apply(normalize_payment_mode)

print("Standardized categorical columns.")

## 7) Invalid Numeric Value Handling
**What:** Convert numeric columns safely, mark invalid values (`<=0`, non-numeric) as missing, then remove unrecoverable rows.

**Why:** Negative/zero quantity or price is not valid for standard transaction analysis.

**What changed:** Invalid numeric rows were handled transparently.

In [ ]:
for col in ["Quantity", "Price", "TotalAmount"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

invalid_quantity_before = ((df["Quantity"] <= 0) | (df["Quantity"].isna())).sum()
invalid_price_before = ((df["Price"] <= 0) | (df["Price"].isna())).sum()

print(f"Invalid Quantity values before handling: {invalid_quantity_before}")
print(f"Invalid Price values before handling: {invalid_price_before}")

# Transparent strategy: mark invalid as NaN and drop rows where they cannot be used for analysis
for col in ["Quantity", "Price"]:
    df.loc[df[col] <= 0, col] = np.nan

df = df.dropna(subset=["Quantity", "Price"]).copy()

invalid_quantity_after = ((df["Quantity"] <= 0) | (df["Quantity"].isna())).sum()
invalid_price_after = ((df["Price"] <= 0) | (df["Price"].isna())).sum()

print(f"Invalid Quantity values after handling: {invalid_quantity_after}")
print(f"Invalid Price values after handling: {invalid_price_after}")

## 8) Date Processing
**What:** Convert `PurchaseDate` to datetime and derive `Month` and `DayOfWeek`.

**Why:** Proper datetime handling is required for trend and seasonality analysis.

**What changed:** Invalid dates removed; new date-derived features added.

In [ ]:
df["PurchaseDate"] = pd.to_datetime(df["PurchaseDate"], errors="coerce")
invalid_dates = df["PurchaseDate"].isna().sum()
print(f"Invalid dates detected: {invalid_dates}")

df = df.dropna(subset=["PurchaseDate"]).copy()

df["MonthNumber"] = df["PurchaseDate"].dt.month
df["Month"] = df["PurchaseDate"].dt.month_name()
df["DayOfWeek"] = df["PurchaseDate"].dt.day_name()

print("Date conversion complete.")

## 9) Feature Engineering
**What:** Recalculate `TotalAmount` for consistency and create `AgeGroup`.

**Why:** Ensures transaction value correctness and enables age-segment insights.

**What changed:** Consistent `TotalAmount` values and grouped age buckets were created.

In [ ]:
calculated_total = df["Quantity"] * df["Price"]

if "TotalAmount" not in df.columns:
    df["TotalAmount"] = calculated_total
else:
    # Replace missing or inconsistent values with calculated totals
    mismatch_mask = df["TotalAmount"].isna() | (~np.isclose(df["TotalAmount"], calculated_total, rtol=1e-5, atol=1e-5))
    df.loc[mismatch_mask, "TotalAmount"] = calculated_total[mismatch_mask]

age_bins = [18, 25, 40, 60, np.inf]
age_labels = ["18-25", "26-40", "41-60", "60+"]
df["AgeGroup"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels, include_lowest=True)

print("Feature engineering complete.")

## 10) Encoding and Transformation
**What:** Demonstrate categorical encoding and min-max normalization.

**Why:** Encoded/scaled columns are useful for machine learning and numeric comparisons.

**What changed:** Added encoded and normalized columns without replacing readable originals.

In [ ]:
# Label-style encoding using pandas factorize (keeps original readable columns)
for col in ["Gender", "City"]:
    encoded_values, _ = pd.factorize(df[col], sort=True)
    df[f"{col}_Encoded"] = encoded_values

def min_max_normalize(series):
    min_val = series.min()
    max_val = series.max()
    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return pd.Series(np.nan, index=series.index)
    return (series - min_val) / (max_val - min_val)

for col in ["Age", "Price", "TotalAmount"]:
    df[f"{col}_Normalized"] = min_max_normalize(df[col])

print("Encoding and normalization completed.")

## 11) Final Data Verification
Verify final data quality conditions after preprocessing.

In [ ]:
print("========== FINAL VERIFICATION ==========")
print("Missing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nInvalid Quantity count (<=0 or NaN):")
print(((df["Quantity"] <= 0) | (df["Quantity"].isna())).sum())

print("\nInvalid Price count (<=0 or NaN):")
print(((df["Price"] <= 0) | (df["Price"].isna())).sum())

print("\nInvalid dates count:")
print(df["PurchaseDate"].isna().sum())

print("\nData types:")
print(df.dtypes)

print("\nTotalAmount consistency check (mismatches):")
print((~np.isclose(df["TotalAmount"], df["Quantity"] * df["Price"], rtol=1e-5, atol=1e-5)).sum())

print("\nAgeGroup value counts (including NaN):")
print(df["AgeGroup"].value_counts(dropna=False))

print("\nMonth and DayOfWeek samples:")
display(df[["PurchaseDate", "Month", "DayOfWeek"]].head())

## 12) Save Cleaned Dataset

In [ ]:
df.to_csv(cleaned_output_path, index=False)
print(f"Cleaned dataset saved to: {cleaned_output_path}")

## KPI Summary
This section reports core metrics directly computed from the cleaned dataset.

In [ ]:
kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Transactions",
        "Total Customers",
        "Total Revenue",
        "Average Transaction Value",
        "Number of Cities",
        "Number of Product Categories"
    ],
    "Value": [
        len(df),
        df["CustomerID"].nunique(dropna=True),
        round(df["TotalAmount"].sum(), 2),
        round(df["TotalAmount"].mean(), 2),
        df["City"].nunique(dropna=True),
        df["ProductCategory"].nunique(dropna=True)
    ]
})

display(kpi_summary)

## 13) Customer Demographics
### Visualization 1: Age Distribution
Shows how customer ages are distributed.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["Age"], bins=20, kde=True, color="steelblue")
plt.title("Customer Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(viz_dir / "age_distribution.png", dpi=300)
plt.show()

### Visualization 2: Gender Distribution
Shows transaction/customer distribution by gender.

In [ ]:
plt.figure(figsize=(8, 5))
gender_counts = df["Gender"].value_counts(dropna=False)
sns.barplot(x=gender_counts.index, y=gender_counts.values, palette="pastel")
plt.title("Gender Distribution")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(viz_dir / "gender_distribution.png", dpi=300)
plt.show()

### Visualization 3: Top 10 Cities
Shows the top cities by transaction count.

In [ ]:
city_counts = df["City"].value_counts().head(10)

plt.figure(figsize=(11, 6))
sns.barplot(x=city_counts.index, y=city_counts.values, palette="viridis")
plt.title("Top 10 Cities by Transactions")
plt.xlabel("City")
plt.ylabel("Transactions")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(viz_dir / "top_10_cities.png", dpi=300)
plt.show()

## 14) Sales Insights
### Visualization 4: Sales by Product Category
Shows total revenue contribution by category.

In [ ]:
category_sales = df.groupby("ProductCategory", dropna=False)["TotalAmount"].sum().sort_values(ascending=False)

plt.figure(figsize=(11, 6))
sns.barplot(x=category_sales.index, y=category_sales.values, palette="magma")
plt.title("Total Sales by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Total Sales (INR)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(viz_dir / "sales_by_category.png", dpi=300)
plt.show()

### Visualization 5: Monthly Sales Trend
Shows month-wise total revenue in chronological order.

In [ ]:
monthly_sales = (
    df.groupby(["MonthNumber", "Month"], dropna=False)["TotalAmount"]
    .sum()
    .reset_index()
    .sort_values("MonthNumber")
)

plt.figure(figsize=(10, 6))
sns.lineplot(data=monthly_sales, x="Month", y="TotalAmount", marker="o", linewidth=2)
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales (INR)")
plt.tight_layout()
plt.savefig(viz_dir / "monthly_sales.png", dpi=300)
plt.show()

### Visualization 6: Payment Mode Usage
Shows distribution of payment methods.

In [ ]:
payment_counts = df["PaymentMode"].value_counts(dropna=False)

plt.figure(figsize=(8, 8))
plt.pie(payment_counts.values, labels=payment_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Payment Mode Usage")
plt.tight_layout()
plt.savefig(viz_dir / "payment_mode.png", dpi=300)
plt.show()

## 15) Advanced Insights
### Visualization 7: Average Spend by Age Group
Shows mean transaction value by age segment.

In [ ]:
age_group_order = ["18-25", "26-40", "41-60", "60+"]
age_group_spend = (
    df.groupby("AgeGroup", dropna=False)["TotalAmount"]
    .mean()
    .reindex(age_group_order)
)

plt.figure(figsize=(9, 5))
sns.barplot(x=age_group_spend.index, y=age_group_spend.values, palette="coolwarm")
plt.title("Average Spend per Customer by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Average Spend (INR)")
plt.tight_layout()
plt.savefig(viz_dir / "age_group_spending.png", dpi=300)
plt.show()

### Visualization 8: City-wise Revenue
Shows top city contributors by revenue (horizontal for readability).

In [ ]:
city_revenue = df.groupby("City", dropna=False)["TotalAmount"].sum().sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 7))
sns.barplot(x=city_revenue.values, y=city_revenue.index, palette="Blues_r")
plt.title("City-wise Revenue Contribution")
plt.xlabel("Total Revenue (INR)")
plt.ylabel("City")
plt.tight_layout()
plt.savefig(viz_dir / "city_revenue.png", dpi=300)
plt.show()

### Visualization 9: Product Category vs Payment Mode Heatmap
Shows cross-pattern of category purchases and payment preference.

In [ ]:
category_payment_ct = pd.crosstab(df["ProductCategory"], df["PaymentMode"])

plt.figure(figsize=(10, 6))
sns.heatmap(category_payment_ct, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Product Category vs Payment Mode")
plt.xlabel("Payment Mode")
plt.ylabel("Product Category")
plt.tight_layout()
plt.savefig(viz_dir / "category_payment_heatmap.png", dpi=300)
plt.show()

## 16) Conclusion
The dataset has been cleaned and transformed into an analysis-ready format. The generated visualizations provide a strong foundation for understanding customer demographics, product-category performance, monthly demand patterns, city-level revenue, and payment behavior. This cleaned dataset can now support further statistical analysis, dashboarding, or machine learning workflows.